In [2]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
import joblib
from sklearn.preprocessing import RobustScaler
from sklearn.mixture import BayesianGaussianMixture
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import classification_report, average_precision_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn import svm
from sklearn.decomposition import PCA, KernelPCA
from sklearn.kernel_approximation import RBFSampler, Nystroem
import umap
import xgboost as xgb


c:\Users\Anton\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
data = pd.read_csv(r'C:\Users\Anton\Downloads\archive (1)\creditcard.csv')
data = data.drop(['Time'], axis=1)

y = data['Class']
X = data.drop(['Class'], axis=1)

# scaling the amount to the other features
X['Amount'] = np.log(X['Amount'] + 0.001)
scaler = RobustScaler()
X['Amount'] = scaler.fit_transform(X[['Amount']])

data


,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,0.090794,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,-0.166974,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,0.207643,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,-0.054952,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,0.753074,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
284802,-11.881118,10.071785,-9.834783,-2.066656,-5.364473,-2.606837,-4.918215,7.305334,1.914428,4.356170,...,0.213454,0.111864,1.014480,-0.509348,1.436807,0.250034,0.943651,0.823731,0.77,0
284803,-0.732789,-0.055080,2.035030,-0.738589,0.868229,1.058415,0.024330,0.294869,0.584800,-0.975926,...,0.214205,0.924384,0.012463,-1.016226,-0.606624,-0.395255,0.068472,-0.053527,24.79,0
284804,1.919565,-0.301254,-3.249640,-0.557828,2.630515,3.031260,-0.296827,0.708417,0.432454,-0.484782,...,0.232045,0.578229,-0.037501,0.640134,0.265745,-0.087371,0.004455,-0.026561,67.88,0
284805,-0.240440,0.530483,0.702510,0.689799,-0.377961,0.623708,-0.686180,0.679145,0.392087,-0.399126,...,0.265245,0.800049,-0.163298,0.123205,-0.569159,0.546668,0.108821,0.104533,10.00,0


In [4]:
model_path = 'bgm_model_antoni.joblib'

if os.path.exists(model_path):
    print("Loading saved model...")
    bgm = joblib.load(model_path)
else:
    bgm = BayesianGaussianMixture(
        n_components=15,
        weight_concentration_prior_type='dirichlet_process',
        weight_concentration_prior=1e-3,
        max_iter=500,
        random_state=42,
        verbose=1
    )
    bgm.fit(X)
    joblib.dump(bgm, model_path)
    print(f"Model saved locally to {model_path}")

Loading saved model...


In [5]:
weights = np.round(bgm.weights_, 4)
active_clusters = np.where(weights > 0.005)[0]

print(f"active clusters: {weights[active_clusters]}")


active clusters: [0.1173 0.0324 0.0513 0.0137 0.3501 0.0557 0.1214 0.007  0.0071 0.0532
 0.0163 0.0096 0.0803 0.045  0.0396]


In [6]:
results = pd.DataFrame()
results['Cluster'] = bgm.predict(X)
results['True_Class'] = y.values 

crosstab = pd.crosstab(results['Cluster'], results['True_Class'])

crosstab = crosstab.rename(columns={0: 'Normal', 1: 'Fraud'})

if 'Fraud' not in crosstab.columns: crosstab['Fraud'] = 0
if 'Normal' not in crosstab.columns: crosstab['Normal'] = 0

crosstab['Fraud_Rate (%)'] = (crosstab['Fraud'] / (crosstab['Normal'] + crosstab['Fraud'])) * 100

print(crosstab.sort_values(by='Fraud_Rate (%)', ascending=False))

True_Class  Normal  Fraud  Fraud_Rate (%)
Cluster                                  
8             1675    346       17.120238
7             1942     61        3.045432
3             3856      7        0.181206
11            2718      4        0.146951
2            14592     10        0.068484
10            4619      3        0.064907
4            99669     39        0.039114
0            33313     11        0.033009
9            15165      5        0.032960
12           22927      4        0.017444
13           12828      1        0.007795
5            15851      1        0.006308
1             9232      0        0.000000
6            34648      0        0.000000
14           11280      0        0.000000


In [7]:
#  We separate 2 good clusters from the rest of the data
best_two_clusters = results[results['Cluster'].isin([7, 8])]
X_best_two_clusters = X.iloc[best_two_clusters.index]
y_best_two_clusters = y.iloc[best_two_clusters.index]

X_worst = X[~X.index.isin(best_two_clusters.index)]
y_worst = y[~y.index.isin(best_two_clusters.index)]

In [8]:
data_worst = pd.concat([X_worst, y_worst], axis=1)
data_worst

worst1 = data_worst[data_worst['Class'] == 1]
worst0 = data_worst[data_worst['Class'] == 0]

# fraudulent data
X_worst1 = worst1.drop(['Class'], axis=1)
y_worst1 = worst1['Class']

# not fraudulent data
X_worst0 = worst0.drop(['Class'], axis=1)
y_worst0 = worst0['Class']

X_worst0_train, X_worst0_test, y_worst0_train, y_worst0_test = train_test_split(X_worst0, y_worst0, test_size=0.2, random_state=42)

## One-class SVM for anomaly detection

In [21]:
clf = svm.OneClassSVM(nu=0.1, kernel="rbf", gamma=0.1)
clf.fit(X_worst0_train)

y_pred_train = clf.predict(X_worst0_train)
y_pred_test = clf.predict(X_worst0_test)
y_pred_outliers = clf.predict(X_worst1)

n_error_train = y_pred_train[y_pred_train == -1].size
n_error_test = y_pred_test[y_pred_test == -1].size
n_error_outliers = y_pred_outliers[y_pred_outliers == 1].size


In [ ]:
X_test_combined = pd.concat([X_worst0_test, X_worst1])
y_test_combined = pd.concat([y_worst0_test, y_worst1])

y_pred_raw = clf.predict(X_test_combined)
y_pred_mapped = np.where(y_pred_raw == 1, 0, 1)

cm = confusion_matrix(y_test_combined, y_pred_mapped)

print("--- Confusion Matrix ---")
print(f"True Negatives (Normal as Normal): {cm[0][0]}")
print(f"False Positives (Normal as Fraud): {cm[0][1]}")
print(f"False Negatives (Fraud as Normal): {cm[1][0]}")
print(f"True Positives  (Fraud as Fraud):  {cm[1][1]}\n")

print("--- Classification Report ---")
print(classification_report(y_test_combined, y_pred_mapped))

--- Confusion Matrix ---
True Negatives (Normal as Normal): 50372
False Positives (Normal as Fraud): 5768
False Negatives (Fraud as Normal): 37
True Positives  (Fraud as Fraud):  48

--- Classification Report ---
              precision    recall  f1-score   support

           0       1.00      0.90      0.95     56140
           1       0.01      0.56      0.02        85

    accuracy                           0.90     56225
   macro avg       0.50      0.73      0.48     56225
weighted avg       1.00      0.90      0.94     56225



* We find the number of false positives to be "fiiine" but significantly better than one optained by the xgm. 
* Number of false positives is more concerning here, as the number of flagged legal transactions (10%) makes the model not very uesful.

## PCA and kernel PCA 

### PCA

In [9]:

for i in range(29):
    pca = PCA(n_components=i+1)
    pca.fit(X_worst)
    print(pca.explained_variance_ratio_.sum())

pca = PCA(n_components=18)
X_worst_top18 = pca.fit_transform(X_worst)

0.12496485985348986
0.22200203811890584
0.2959307444987723
0.36415749816495946
0.42757290623606126
0.487750888039131
0.532657033108326
0.574372956785679
0.6136026552853672
0.6497883105178448
0.6853541669614166
0.7195361222082558
0.7515889440063424
0.7805655707240182
0.808332315090984
0.8341829049499472
0.8584755820359146
0.8813178762059104
0.9024162487397754
0.9208058464516401
0.9360301228177174
0.949216742600622
0.9619335207556726
0.9739255932832882
0.983242588336983
0.9913072174374226
0.9963594610397049
0.9999991430083822
0.9999999999999999


In [12]:
X_worst_top18_df = pd.DataFrame(X_worst_top18, columns=[f'PC{i+1}' for i in range(18)])
X_worst_top18_df

,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,PC11,PC12,PC13,PC14,PC15,PC16,PC17,PC18
0,-1.708951,0.427932,2.574255,0.752552,-0.322054,0.410973,0.339418,0.270531,0.076177,-0.182374,-1.060552,-0.870279,-0.240343,1.380235,-0.315066,0.640120,0.151001,0.432751
1,1.201792,0.336654,0.290809,0.412935,0.224556,-0.008591,0.088785,-0.111748,-0.074526,1.161684,1.586061,0.452370,-0.291640,0.721364,0.364014,-0.358780,-0.216828,-0.118502
2,-1.893874,-1.084839,1.838406,-0.123325,-1.071010,1.534784,1.250741,-1.420051,-0.252096,0.381880,0.362391,0.670274,-0.097001,2.176576,-1.900581,2.645648,-0.071413,-2.200688
3,-1.322244,0.132903,1.239064,-1.084870,-0.145891,1.494785,0.933909,-1.219322,-0.083769,-0.459167,0.199741,0.437784,-0.324703,-0.694826,-0.822341,0.201147,2.079497,-1.286458
4,-1.252930,1.165260,1.268955,0.016861,-0.584172,-0.172816,-0.041726,0.378354,-0.345332,-0.347775,-0.675469,1.563376,-1.588283,0.075173,-0.354178,0.340030,0.202300,0.798583
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
280778,-8.141667,8.339930,-8.474402,-2.520647,-5.220056,-4.433097,-12.648645,-1.275177,8.837570,-1.229931,1.841158,0.174744,3.261643,-1.046229,-1.384582,-0.240244,-0.533180,-0.576316
280779,-1.078827,0.574307,1.134903,-0.630072,0.764817,1.713736,0.885878,0.987099,-0.196521,-0.414262,0.627567,1.238358,-0.985712,1.120874,-0.662242,0.442587,-1.069304,-1.515469
280780,2.007272,-1.087896,-3.612079,0.863783,-0.274574,3.452188,0.033356,0.546324,0.201037,0.123120,0.372828,-0.189155,-0.049766,1.316134,0.410006,0.523027,0.481757,-0.594839
280781,-0.203618,0.647165,0.935132,0.376247,-0.316523,0.611312,-0.689783,0.487973,0.716199,-1.705244,-1.249385,-1.021016,0.813378,1.861082,-0.466640,0.748036,1.078838,2.904318


### Kernel PCA

We note that running a full kernel pca on this dataset would require 700GB of RAM, so we opt for approximate methods instead

#### Random Fourier Features

In [13]:
rbf_features = RBFSampler(
    gamma=1.0,
    n_components=1000,
    random_state=42
)

Z = rbf_features.fit_transform(X_worst)


#for i in range(1000):
#    p = PCA(n_components=i+1)
#    p.fit(Z)
#    print(p.explained_variance_ratio_.sum())

In [14]:
p = PCA(n_components=400)
p.fit(Z)
print(p.explained_variance_ratio_.sum())

0.49222488055146235


* We tried to approximate with 1000 componants, and we see that it is quite horrible at reducing dimentionality. That being said, we still tried running xgb on the whole thing. Just for fun. I will not attempt runnning a svm on a 280783 x 1000 dataset for obvious reasons. Maybe i will run a sgd svm at some point. We will see.

In [ ]:
pca = PCA(n_components=400)
X_reduced = pca.fit_transform(Z)


array([[ 0.00025595, -0.02264003,  0.00232027, ...,  0.0216444 ,
        -0.03063527, -0.00684846],
       [ 0.02980459, -0.08291739,  0.1508085 , ...,  0.0309726 ,
         0.03327982, -0.00739905],
       [-0.0185788 ,  0.01507961, -0.00718783, ..., -0.04719752,
        -0.00860597, -0.0285719 ],
       ...,
       [ 0.03416428,  0.03347587,  0.04704401, ..., -0.0310434 ,
        -0.00667115,  0.01291016],
       [-0.02996598, -0.01073149,  0.02621474, ..., -0.0005991 ,
        -0.0076757 ,  0.02988848],
       [ 0.00548342,  0.01195489, -0.01466348, ...,  0.01738744,
         0.0110525 , -0.01208663]], shape=(280783, 400))

In [18]:
X_reduced_df = pd.DataFrame(X_reduced, columns=[f'PC{i+1}' for i in range(400)])
X_reduced_df

,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,...,PC391,PC392,PC393,PC394,PC395,PC396,PC397,PC398,PC399,PC400
0,0.000256,-0.022640,0.002320,0.018021,-0.048538,0.007690,-0.028864,0.010702,0.002399,0.008965,...,-0.016063,-0.017773,-0.013430,0.029556,0.013232,-0.019558,0.058839,0.021644,-0.030635,-0.006848
1,0.029805,-0.082917,0.150808,0.070336,0.002717,-0.083768,0.317138,0.074700,0.069284,-0.008836,...,0.020031,0.002933,0.008661,-0.055614,-0.016158,0.007463,0.022742,0.030973,0.033280,-0.007399
2,-0.018579,0.015080,-0.007188,0.003715,0.004383,0.040886,-0.046992,-0.085459,0.019364,0.015944,...,-0.036037,-0.046413,-0.001111,0.001056,0.020605,-0.016183,0.008380,-0.047198,-0.008606,-0.028572
3,-0.017677,-0.010691,-0.036305,0.032341,-0.038224,0.016929,-0.026019,0.000285,0.022415,0.046118,...,-0.064393,-0.009033,0.023494,-0.028892,0.001455,0.051671,-0.066171,-0.024084,0.008352,-0.013815
4,-0.019186,0.009571,0.001271,0.044763,0.021471,0.071264,0.025796,-0.003285,-0.038315,-0.010183,...,0.013202,0.016150,-0.006350,0.023103,0.019079,-0.021198,0.012978,0.011111,0.039692,0.025535
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
280778,-0.061451,0.006013,0.057791,-0.039677,0.004058,0.022141,-0.028689,-0.019281,-0.002179,-0.017712,...,0.045226,0.027826,0.002383,-0.006993,-0.044270,-0.027415,0.041437,0.009018,-0.012122,0.015906
280779,-0.003383,0.003081,0.002581,0.004441,0.055706,0.021056,0.036514,-0.015105,-0.039421,-0.015037,...,-0.006996,0.051498,-0.013039,-0.005797,-0.005694,-0.090296,0.004834,0.038047,-0.055114,0.066896
280780,0.034164,0.033476,0.047044,0.021348,0.019699,-0.025457,0.067331,-0.048663,0.030263,0.035013,...,-0.018321,-0.006950,0.013311,0.011452,0.013850,-0.012403,-0.030487,-0.031043,-0.006671,0.012910
280781,-0.029966,-0.010731,0.026215,-0.040529,-0.023881,0.015163,0.010053,0.038832,-0.028912,0.004677,...,-0.034640,-0.043934,0.014015,-0.017276,-0.054512,-0.009681,-0.029918,-0.000599,-0.007676,0.029888


In [ ]:
X_train_reduced, X_test_reduced, y_train_worst, y_test_worst = train_test_split(X_reduced_df, y_worst, test_size=0.2, random_state=42, stratify=y_worst)

neg_class_count = (y_train_worst == 0).sum()
pos_class_count = (y_train_worst == 1).sum()
imbalance_ratio = neg_class_count / pos_class_count

xgb_model = xgb.XGBClassifier(
    scale_pos_weight=imbalance_ratio,
    max_depth=4,
    learning_rate=0.05,
    n_estimators=300,
    objective='binary:logistic',
    random_state=42,                
    n_jobs=-1
)

xgb_model.fit(X_train_reduced, y_train_worst)

y_pred = xgb_model.predict(X_test_reduced)
y_pred_proba = xgb_model.predict_proba(X_test_reduced)[:, 1]

# Evaluation
print("\n--- Confusion Matrix ---")
cm = confusion_matrix(y_test_worst, y_pred)
print(f"True Negatives: {cm[0][0]}  |  False Positives: {cm[0][1]}")
print(f"False Negatives: {cm[1][0]}    |  True Positives: {cm[1][1]}")

print("\n--- Classification Report ---")
print(classification_report(y_test_worst, y_pred))

auprc = average_precision_score(y_test_worst, y_pred_proba)
print(f"Area Under the Precision-Recall Curve (AUPRC): {auprc:.4f}")



--- Confusion Matrix ---
True Negatives: 56140  |  False Positives: 0
False Negatives: 17    |  True Positives: 0

--- Classification Report ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56140
           1       0.00      0.00      0.00        17

    accuracy                           1.00     56157
   macro avg       0.50      0.50      0.50     56157
weighted avg       1.00      1.00      1.00     56157

Area Under the Precision-Recall Curve (AUPRC): 0.0004


c:\Users\Anton\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Anton\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Anton\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(ave

* Whis is so shit we could get the same result with def model: return 0

In [15]:
X_train_top18, X_test_top18, y_train_worst, y_test_worst = train_test_split(X_worst_top18_df, y_worst, test_size=0.2, random_state=42, stratify=y_worst)

neg_class_count = (y_train_worst == 0).sum()
pos_class_count = (y_train_worst == 1).sum()
imbalance_ratio = neg_class_count / pos_class_count

xgb_model = xgb.XGBClassifier(
    scale_pos_weight=imbalance_ratio,
    max_depth=4,
    learning_rate=0.05,
    n_estimators=300,
    objective='binary:logistic',
    random_state=42,                
    n_jobs=-1
)

xgb_model.fit(X_train_top18, y_train_worst)

y_pred = xgb_model.predict(X_test_top18)
y_pred_proba = xgb_model.predict_proba(X_test_top18)[:, 1]

# Evaluation
print("\n--- Confusion Matrix ---")
cm = confusion_matrix(y_test_worst, y_pred)
print(f"True Negatives: {cm[0][0]}  |  False Positives: {cm[0][1]}")
print(f"False Negatives: {cm[1][0]}    |  True Positives: {cm[1][1]}")

print("\n--- Classification Report ---")
print(classification_report(y_test_worst, y_pred))

auprc = average_precision_score(y_test_worst, y_pred_proba)
print(f"Area Under the Precision-Recall Curve (AUPRC): {auprc:.4f}")



--- Confusion Matrix ---
True Negatives: 56028  |  False Positives: 112
False Negatives: 15    |  True Positives: 2

--- Classification Report ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56140
           1       0.02      0.12      0.03        17

    accuracy                           1.00     56157
   macro avg       0.51      0.56      0.51     56157
weighted avg       1.00      1.00      1.00     56157

Area Under the Precision-Recall Curve (AUPRC): 0.0760


* It is hard to say if it is better, shittier, the same or just different. I will leave the interpretation to the reader